## Cell 1 — Setup

In [1]:
from google.colab import drive
drive.mount('/content/drive')
import os
os.chdir('/content/drive/MyDrive/ML_final')

import pandas as pd
import numpy as np

train = pd.read_pickle('data/train_processed.pkl')
test = pd.read_pickle('data/test_processed.pkl')

!pip install -q dagshub mlflow

import dagshub
dagshub.init(repo_owner='skupr23', repo_name='ml_final', mlflow=True)

import mlflow

EXPERIMENT_NAME = 'Ensemble_Training'
REGISTERED_MODEL_NAME = 'WalmartSalesForecast'

mlflow.set_experiment(EXPERIMENT_NAME)
if mlflow.active_run() is not None:
    mlflow.end_run()

print('Tracking URI:', mlflow.get_tracking_uri())
print('Experiment  :', EXPERIMENT_NAME)

Mounted at /content/drive
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.7/49.7 kB 3.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.5/50.5 kB 5.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 273.1/273.1 kB 18.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 94.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 88.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 74.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 264.7/264.7 kB 24.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 94.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.2/68.2 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 148.8/148.8 kB 15.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━

❗❗❗ AUTHORIZATION REQUIRED ❗❗❗

Output()



Open the following link in your browser to authorize the client:
https://dagshub.com/login/oauth/authorize?state=c1f54af4-28b7-40ed-a860-b7954471f784&client_id=32b60ba385aa7cecf24046d8195a71c07dd345d9657977863b52e7748e0f0f28&middleman_request_id=4f774ec8cdc98b277fbf132cd6b11a50b00e2bd09dc5b7b6c6aec0baafcb4c8d




Accessing as ggzob23

Initialized MLflow to track repo "skupr23/ml_final"

Repository skupr23/ml_final initialized!

Tracking URI: https://dagshub.com/skupr23/ml_final.mlflow
Experiment  : Ensemble_Training


## Cell 2 — WMAE Metric

In [2]:
def wmae(y_true, y_pred, is_holiday):
    weights = np.where(is_holiday, 5, 1)
    return np.sum(weights * np.abs(y_true - y_pred)) / np.sum(weights)

## Cell 3 — Chronological Validation Split + Baseline

In [3]:
train = train.sort_values('Date')
val = train[train['Date'] >= train['Date'].max() - pd.Timedelta(weeks=39)].copy()
fit = train[train['Date'] < val['Date'].min()].copy()
print('fit ends:', fit['Date'].max())
print('val range:', val['Date'].min(), 'to', val['Date'].max())

baseline_pred = val['lag_52'].fillna(val['storedept_median_sales'])
baseline_score = wmae(val['Weekly_Sales'], baseline_pred, val['IsHoliday'])
print('52-week seasonal baseline WMAE:', baseline_score)

fit ends: 2012-01-20 00:00:00
val range: 2012-01-27 00:00:00 to 2012-10-26 00:00:00
52-week seasonal baseline WMAE: 1789.9133525504185


## Cell 4 — Prepare Features (Excluding Leaky Lags)

In [4]:
cat_cols = ['Store','Dept','Type','holiday_type']
for df in [fit, val, test, train]:
    for c in cat_cols:
        df[c] = df[c].astype('category')

UNAVAILABLE_AT_INFERENCE = ['lag_1', 'lag_2', 'lag_4', 'lag_13', 'lag_26', 'sales_change_1']
drop_cols = ['Weekly_Sales','Date','Id'] if 'Id' in fit.columns else ['Weekly_Sales','Date']
drop_cols += UNAVAILABLE_AT_INFERENCE
feature_cols = [c for c in fit.columns if c not in drop_cols]

X_fit, y_fit = fit[feature_cols], fit['Weekly_Sales']
X_val, y_val = val[feature_cols], val['Weekly_Sales']
print('Total features going in:', len(feature_cols))

Total features going in: 42


## Cell 5 — Fit XGBoost and LightGBM on Fit-Only Data (Blend Validation)

In [5]:
import xgboost as xgb
import lightgbm as lgb

model_xgb_val = xgb.XGBRegressor(
    objective='reg:absoluteerror',
    n_estimators=2000,
    learning_rate=0.03,
    max_depth=8,
    device='cuda',
    enable_categorical=True,
    early_stopping_rounds=50,
    random_state=42,
    eval_metric='mae'
)
model_xgb_val.fit(X_fit, y_fit, eval_set=[(X_val, y_val)], verbose=100)

model_lgb_val = lgb.LGBMRegressor(
    objective='mae', n_estimators=2000, learning_rate=0.03,
    num_leaves=64, random_state=42
)
model_lgb_val.fit(
    X_fit, y_fit,
    eval_set=[(X_val, y_val)],
    categorical_feature=cat_cols,
    callbacks=[lgb.early_stopping(50), lgb.log_evaluation(200)]
)

pred_xgb_val = model_xgb_val.predict(X_val)
pred_lgb_val = model_lgb_val.predict(X_val)

wmae_xgb = wmae(y_val, pred_xgb_val, val['IsHoliday'])
wmae_lgb = wmae(y_val, pred_lgb_val, val['IsHoliday'])
print('XGBoost alone WMAE:', wmae_xgb)
print('LightGBM alone WMAE:', wmae_lgb)

[0]	validation_0-mae:12973.64383
[100]	validation_0-mae:1973.49170
[200]	validation_0-mae:1446.49410
[300]	validation_0-mae:1412.61362
[400]	validation_0-mae:1398.77357
[500]	validation_0-mae:1380.05445
[600]	validation_0-mae:1365.69075
[700]	validation_0-mae:1359.89211
[800]	validation_0-mae:1356.83822
[900]	validation_0-mae:1352.33990
[1000]	validation_0-mae:1348.67971
[1100]	validation_0-mae:1346.89236
[1200]	validation_0-mae:1345.38159
[1300]	validation_0-mae:1343.95609
[1400]	validation_0-mae:1343.28637
[1500]	validation_0-mae:1343.51619
[1508]	validation_0-mae:1343.37911
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.083203 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 5502
[LightGBM] [Info] Number of data points in the train set: 303030, number of used features: 40
[LightGBM] [Info] Start training from score 7679.899902
Traini

/usr/local/lib/python3.12/dist-packages/xgboost/core.py:553: UserWarning: [12:55:05] WARNING: /__w/xgboost/xgboost/src/common/error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


XGBoost alone WMAE: 1360.7025373776128
LightGBM alone WMAE: 1384.8175014866931


## Cell 6 — Grid Search the Blend Ratio

In [6]:
best_ratio = None
best_blend_wmae = float('inf')
blend_results = []

for ratio in np.arange(0.0, 1.01, 0.05):
    blend_pred = ratio * pred_xgb_val + (1 - ratio) * pred_lgb_val
    w = wmae(y_val, blend_pred, val['IsHoliday'])
    blend_results.append({'xgb_weight': ratio, 'wmae': w})
    if w < best_blend_wmae:
        best_blend_wmae = w
        best_ratio = ratio

blend_results_df = pd.DataFrame(blend_results)
print(blend_results_df)
print(f'\nBest blend: {best_ratio:.2f} XGBoost / {1-best_ratio:.2f} LightGBM -> WMAE: {best_blend_wmae:.2f}')
print('vs XGBoost alone:  ', wmae_xgb)
print('vs LightGBM alone: ', wmae_lgb)
print('vs baseline:       ', baseline_score)

    xgb_weight         wmae
0         0.00  1384.817501
1         0.05  1380.664029
2         0.10  1376.820717
3         0.15  1373.251165
4         0.20  1370.000893
5         0.25  1367.041871
6         0.30  1364.385609
7         0.35  1362.059872
8         0.40  1360.056996
9         0.45  1358.391614
10        0.50  1357.040087
11        0.55  1355.996169
12        0.60  1355.276971
13        0.65  1354.855868
14        0.70  1354.773411
15        0.75  1354.969302
16        0.80  1355.485329
17        0.85  1356.310833
18        0.90  1357.442440
19        0.95  1358.918055
20        1.00  1360.702537

Best blend: 0.70 XGBoost / 0.30 LightGBM -> WMAE: 1354.77
vs XGBoost alone:   1360.7025373776128
vs LightGBM alone:  1384.8175014866931
vs baseline:        1789.9133525504185


## Cell 7 — Holiday vs Non-Holiday Breakdown for the Best Blend

In [7]:
best_blend_pred = best_ratio * pred_xgb_val + (1 - best_ratio) * pred_lgb_val
is_hol = val['IsHoliday'].values

print('Holiday WMAE:    ', wmae(y_val[is_hol], best_blend_pred[is_hol], val['IsHoliday'][is_hol]))
print('Non-holiday WMAE:', wmae(y_val[~is_hol], best_blend_pred[~is_hol], val['IsHoliday'][~is_hol]))

Holiday WMAE:     1442.0563042019683
Non-holiday WMAE: 1331.6409976209654


## Cell 8 — Retrain Both Models on Full Labeled Data

In [8]:
X_full, y_full = train[feature_cols], train['Weekly_Sales']

model_xgb_final = xgb.XGBRegressor(
    objective='reg:absoluteerror',
    n_estimators=int(model_xgb_val.best_iteration * 1.1),
    learning_rate=0.03,
    max_depth=8,
    device='cuda',
    enable_categorical=True,
    random_state=42,
    eval_metric='mae'
)
model_xgb_final.fit(X_full, y_full, verbose=100)

model_lgb_final = lgb.LGBMRegressor(
    objective='mae',
    n_estimators=int(model_lgb_val.best_iteration_ * 1.1),
    learning_rate=0.03,
    num_leaves=64,
    random_state=42,
)
model_lgb_final.fit(X_full, y_full, categorical_feature=cat_cols)

print('Both base models retrained on full data:', len(X_full), 'rows')

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.066529 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 5568
[LightGBM] [Info] Number of data points in the train set: 421570, number of used features: 40
[LightGBM] [Info] Start training from score 7612.029785
Both base models retrained on full data: 421570 rows


## Cell 9 — Save Models Locally

In [9]:
import os, joblib
os.makedirs('models', exist_ok=True)
joblib.dump(model_xgb_final, 'models/ensemble_xgb.pkl')
joblib.dump(model_lgb_final, 'models/ensemble_lgb.pkl')
print('Saved.')

Saved.


## Cell 10 — Raw-Input Pipeline Components (WalmartFeatureEngineer, CategoricalCaster, FeatureSelector)

In [10]:
import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline

MARKDOWN_COLS = ['MarkDown1', 'MarkDown2', 'MarkDown3', 'MarkDown4', 'MarkDown5']
LAG_WEEKS = [1, 2, 4, 13, 26, 51, 52, 53]
ROLL_COLS = ['rolling_mean_4', 'rolling_median_8', 'rolling_mean_13', 'rolling_mean_26']

SUPERBOWL = ['2010-02-12', '2011-02-11', '2012-02-10', '2013-02-08']
LABORDAY = ['2010-09-10', '2011-09-09', '2012-09-07', '2013-09-06']
THANKSGIVING = ['2010-11-26', '2011-11-25', '2012-11-23', '2013-11-29']
CHRISTMAS = ['2010-12-31', '2011-12-30', '2012-12-28', '2013-12-27']

def holiday_type_of(date):
    d = date.strftime('%Y-%m-%d')
    if d in SUPERBOWL: return 'SuperBowl'
    if d in LABORDAY: return 'LaborDay'
    if d in THANKSGIVING: return 'Thanksgiving'
    if d in CHRISTMAS: return 'Christmas'
    return 'Normal'

class WalmartFeatureEngineer(BaseEstimator, TransformerMixin):
    def __init__(self, features, stores, lag_weeks=tuple(LAG_WEEKS)):
        self.features = features
        self.stores = stores
        self.lag_weeks = lag_weeks

    def fit(self, X, y=None):
        raw = X.copy()
        raw['Date'] = pd.to_datetime(raw['Date'])
        if y is not None:
            raw['Weekly_Sales'] = np.asarray(y)
        raw = raw.sort_values(['Store', 'Dept', 'Date']).reset_index(drop=True)

        self.history_ = raw[['Store', 'Dept', 'Date', 'Weekly_Sales']].copy()
        self.train_keys_ = raw[['Store', 'Date']].drop_duplicates()

        rs = self.history_.copy()
        g = rs.groupby(['Store', 'Dept'])['Weekly_Sales']
        rs['rolling_mean_4'] = g.shift(1).rolling(4).mean().reset_index(drop=True)
        rs['rolling_median_8'] = g.shift(1).rolling(8).median().reset_index(drop=True)
        rs['rolling_mean_13'] = g.shift(1).rolling(13).mean().reset_index(drop=True)
        rs['rolling_mean_26'] = g.shift(1).rolling(26).mean().reset_index(drop=True)

        self.last_roll_ = (rs.sort_values('Date')
                             .groupby(['Store', 'Dept'])[ROLL_COLS]
                             .last().reset_index())

        self.dept_median_ = raw.groupby('Dept')['Weekly_Sales'].median().rename('dept_median_sales')
        self.store_median_ = raw.groupby('Store')['Weekly_Sales'].median().rename('store_median_sales')
        self.storedept_median_ = (raw.groupby(['Store', 'Dept'])['Weekly_Sales']
                                     .median().rename('storedept_median_sales'))
        return self

    def _macro(self, test_keys):
        combined = pd.concat([self.train_keys_, test_keys], ignore_index=True).drop_duplicates()
        combined = combined.merge(self.features[['Store', 'Date', 'CPI', 'Unemployment']],
                                  on=['Store', 'Date'], how='left')
        combined = combined.sort_values(['Store', 'Date'])
        for col in ['CPI', 'Unemployment']:
            combined[f'{col}_missing'] = combined[col].isna().astype('int8')
            combined[col] = combined.groupby('Store')[col].ffill()
        return combined

    def transform(self, X):
        df = X.copy()
        df['Date'] = pd.to_datetime(df['Date'])
        df = df.sort_values(['Store', 'Dept', 'Date']).reset_index(drop=True)

        df = (df.merge(self.features, on=['Store', 'Date', 'IsHoliday'],
                       how='left', validate='many_to_one')
                .merge(self.stores, on='Store', how='left', validate='many_to_one'))

        for col in MARKDOWN_COLS:
            df[f'{col}_missing'] = df[col].isna().astype('int8')
            df[col] = df[col].fillna(0)
        df['markdown_total'] = df[MARKDOWN_COLS].sum(axis=1)
        df['markdown_count'] = (df[MARKDOWN_COLS] > 0).sum(axis=1)
        df['has_any_markdown'] = (df['markdown_total'] > 0).astype('int8')

        macro = self._macro(df[['Store', 'Date']].drop_duplicates())
        df = df.drop(columns=['CPI', 'Unemployment']).merge(macro, on=['Store', 'Date'], how='left')

        df['holiday_type'] = df['Date'].apply(holiday_type_of)
        df['year'] = df['Date'].dt.year
        df['month'] = df['Date'].dt.month
        df['week_of_year'] = df['Date'].dt.isocalendar().week.astype(int)
        df['week_sin'] = np.sin(2 * np.pi * df['week_of_year'] / 52)
        df['week_cos'] = np.cos(2 * np.pi * df['week_of_year'] / 52)

        for w in self.lag_weeks:
            shifted = self.history_.copy()
            shifted['Date'] = shifted['Date'] + pd.Timedelta(weeks=w)
            shifted = shifted.rename(columns={'Weekly_Sales': f'lag_{w}'})
            df = df.merge(shifted, on=['Store', 'Dept', 'Date'], how='left')

        df = df.merge(self.last_roll_, on=['Store', 'Dept'], how='left')
        df['sales_change_1'] = df['lag_1'] - df['lag_2']
        df['sales_per_store_size'] = df['lag_52'] / df['Size']
        df['markdown_per_store_size'] = df['markdown_total'] / df['Size']

        df = (df.merge(self.dept_median_, on='Dept', how='left')
                .merge(self.store_median_, on='Store', how='left')
                .merge(self.storedept_median_, on=['Store', 'Dept'], how='left'))
        return df

class CategoricalCaster(BaseEstimator, TransformerMixin):
    def __init__(self, cat_cols):
        self.cat_cols = cat_cols
    def fit(self, X, y=None):
        return self
    def transform(self, X):
        X = X.copy()
        for c in self.cat_cols:
            if c in X.columns:
                X[c] = X[c].astype('category')
        return X

class FeatureSelector(BaseEstimator, TransformerMixin):
    def __init__(self, features):
        self.features = features
    def fit(self, X, y=None):
        return self
    def transform(self, X):
        return X[self.features]

## Cell 11 — Ensemble Estimator (BlendedEstimator)

In [11]:
from sklearn.base import BaseEstimator, RegressorMixin

class BlendedEstimator(BaseEstimator, RegressorMixin):
    def __init__(self, model_xgb, model_lgb, xgb_weight):
        self.model_xgb = model_xgb
        self.model_lgb = model_lgb
        self.xgb_weight = xgb_weight

    def fit(self, X, y=None):
        return self

    def predict(self, X):
        pred_xgb = self.model_xgb.predict(X)
        pred_lgb = self.model_lgb.predict(X)
        return self.xgb_weight * pred_xgb + (1 - self.xgb_weight) * pred_lgb


def build_full_pipeline(blended_model, model_features, feature_engineer):
    return Pipeline([
        ('feature_engineering', feature_engineer),
        ('cast_categoricals', CategoricalCaster(cat_cols)),
        ('select_features', FeatureSelector(model_features)),
        ('model', blended_model),
    ])

## Cell 12 — Fit Feature Engineer on Raw Data, Build Final Ensemble Pipeline

In [12]:
train_raw = pd.read_csv('data/train.csv', parse_dates=['Date'])
test_raw = pd.read_csv('data/test.csv', parse_dates=['Date'])
features_raw = pd.read_csv('data/features.csv', parse_dates=['Date'])
stores_raw = pd.read_csv('data/stores.csv')

feature_engineer = WalmartFeatureEngineer(features_raw, stores_raw).fit(
    train_raw.drop(columns='Weekly_Sales'),
    train_raw['Weekly_Sales'],
)

blended_model = BlendedEstimator(model_xgb_final, model_lgb_final, xgb_weight=best_ratio)
final_pipeline = build_full_pipeline(blended_model, feature_cols, feature_engineer)
print('Ensemble pipeline ready | xgb_weight:', best_ratio, '| lgb_weight:', 1 - best_ratio)

Ensemble pipeline ready | xgb_weight: 0.7000000000000001 | lgb_weight: 0.29999999999999993


## Cell 13 — Sanity Check

In [13]:
_ref = test.sort_values(['Store', 'Dept', 'Date']).reset_index(drop=True)
_ref_pred = (best_ratio * model_xgb_final.predict(_ref[feature_cols])
             + (1 - best_ratio) * model_lgb_final.predict(_ref[feature_cols]))
_pipe_pred = final_pipeline.predict(test_raw)

pipeline_max_abs_diff = float(np.abs(_ref_pred - _pipe_pred).max())
print('max |pipeline(raw) - blend(processed)| =', pipeline_max_abs_diff)
assert pipeline_max_abs_diff < 1e-6, 'pipeline does not reproduce the blended predictions'

max |pipeline(raw) - blend(processed)| = 0.0


/usr/local/lib/python3.12/dist-packages/sklearn/pipeline.py:62: FutureWarning: This Pipeline instance is not fitted yet. Call 'fit' with appropriate arguments before using other methods such as transform, predict, etc. This will raise an error in 1.8 instead of the current warning.
  warnings.warn(


## Cell 14 — MLflow: Baseline, Blend Search, Validation, Final Model

In [14]:
import tempfile
import mlflow.sklearn

with mlflow.start_run(run_name='Ensemble_Baseline'):
    mlflow.set_tag('stage', 'baseline')
    mlflow.log_metric('val_wmae', baseline_score)

with mlflow.start_run(run_name='Ensemble_Blend_Search'):
    mlflow.set_tag('stage', 'blend_search')
    mlflow.log_params({
        'xgb_wmae_alone': wmae_xgb,
        'lgb_wmae_alone': wmae_lgb,
        'search_range': '0.0 to 1.0 step 0.05',
    })
    mlflow.log_metrics({
        'best_xgb_weight': best_ratio,
        'best_blend_wmae': best_blend_wmae,
    })
    with tempfile.TemporaryDirectory() as tmp:
        p = os.path.join(tmp, 'blend_search_results.csv')
        blend_results_df.to_csv(p, index=False)
        mlflow.log_artifact(p, artifact_path='blend_search')

with mlflow.start_run(run_name='Ensemble_Validation'):
    mlflow.set_tag('stage', 'validation')
    mlflow.log_params({
        'scheme': 'chronological holdout, identical to XGBoost/LightGBM notebooks',
        'val_weeks': 39,
        'xgb_weight': best_ratio,
        'lgb_weight': 1 - best_ratio,
    })
    mlflow.log_metrics({
        'baseline_wmae': baseline_score,
        'xgb_alone_wmae': wmae_xgb,
        'lgb_alone_wmae': wmae_lgb,
        'ensemble_wmae': best_blend_wmae,
        'improvement_over_baseline': baseline_score - best_blend_wmae,
        'improvement_over_best_single_model': min(wmae_xgb, wmae_lgb) - best_blend_wmae,
    })

with mlflow.start_run(run_name='Ensemble_Final_Model') as final_run:
    mlflow.set_tags({'stage': 'final_model', 'algorithm': 'xgboost_lightgbm_blend'})
    mlflow.log_params({
        'xgb_weight': best_ratio,
        'lgb_weight': 1 - best_ratio,
        'n_features': len(feature_cols),
    })
    mlflow.log_metrics({
        'final_wmae': best_blend_wmae,
        'baseline_wmae': baseline_score,
        'pipeline_vs_processed_max_abs_diff': pipeline_max_abs_diff,
    })

    model_info = mlflow.sklearn.log_model(
        final_pipeline,
        artifact_path='pipeline',
        serialization_format=mlflow.sklearn.SERIALIZATION_FORMAT_CLOUDPICKLE,
        input_example=test_raw.head(5),
    )
    mlflow.log_artifact('models/ensemble_xgb.pkl', artifact_path='estimator')
    mlflow.log_artifact('models/ensemble_lgb.pkl', artifact_path='estimator')

    FINAL_RUN_ID = final_run.info.run_id
    FINAL_MODEL_URI = model_info.model_uri

print('Ensemble final run:', FINAL_RUN_ID)
print('Model URI for registration:', FINAL_MODEL_URI)

🏃 View run Ensemble_Baseline at: https://dagshub.com/skupr23/ml_final.mlflow/#/experiments/11/runs/37ebc51eb93346b9bb2a68bd65bdc7d6
🧪 View experiment at: https://dagshub.com/skupr23/ml_final.mlflow/#/experiments/11
🏃 View run Ensemble_Blend_Search at: https://dagshub.com/skupr23/ml_final.mlflow/#/experiments/11/runs/aca1444bc65547dd97238c835fbb9331
🧪 View experiment at: https://dagshub.com/skupr23/ml_final.mlflow/#/experiments/11
🏃 View run Ensemble_Validation at: https://dagshub.com/skupr23/ml_final.mlflow/#/experiments/11/runs/a0bc8314995a48d8b7d81d94f192f4c1
🧪 View experiment at: https://dagshub.com/skupr23/ml_final.mlflow/#/experiments/11


2026/07/12 12:59:59 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/07/12 13:00:02 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
/usr/local/lib/python3.12/dist-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can 

🏃 View run Ensemble_Final_Model at: https://dagshub.com/skupr23/ml_final.mlflow/#/experiments/11/runs/b0c239b60ef446489d3c7f853e9f5b5b
🧪 View experiment at: https://dagshub.com/skupr23/ml_final.mlflow/#/experiments/11
Ensemble final run: b0c239b60ef446489d3c7f853e9f5b5b
Model URI for registration: models:/m-609a0b5a4ebc4354ba7af3f03e624b26


## Cell 15 — Registration

In [15]:
REGISTER_AS_GLOBAL_BEST = True

if REGISTER_AS_GLOBAL_BEST:
    registered_version = mlflow.register_model(model_uri=FINAL_MODEL_URI, name=REGISTERED_MODEL_NAME)
    print('Registered model:', REGISTERED_MODEL_NAME, '| version:', registered_version.version)
else:
    print('Pipeline logged and saved, not registered. Register only if the ensemble wins overall.')

Registered model 'WalmartSalesForecast' already exists. Creating a new version of this model...
2026/07/12 13:00:32 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: WalmartSalesForecast, version 9
Created version '9' of model 'WalmartSalesForecast'.


Registered model: WalmartSalesForecast | version: 9
